# Word Similarities

### **Task**
Find the similarity between two words.

### **Objectives**
- Compare word pairs using cosine similarity, Jaccard similarity, and Euclidean distance
- Show how semantic similarity differs from spelling-based similarity
- Analyze how embedding dimensionality affects similarity scores

### About the Pre-trained Model: GloVe (Global Vectors for Word Representation)

In this notebook, we are using **GloVe**, an unsupervised learning algorithm developed by Stanford for obtaining vector representations (embeddings) for words. 

Instead of training our own model from scratch, we are loading pre-trained models via `gensim`. Specifically, we are using the `glove-wiki-gigaword` models, which were trained on a massive combined dataset of Wikipedia and Gigaword (billions of tokens).

We are loading four variations of this model: **50, 100, 200, and 300 dimensions**. 
* **Dimensions** refer to the length of the numeric array used to represent each word. 
* Higher dimensions generally capture more nuanced, fine-grained semantic relationships between words, but they require more memory and computational power to process. By comparing across all four, we can observe how the vector size affects the similarity scores and distances.

In [28]:
import gensim.downloader as api
import numpy as np

# 1. Load the Pre-trained GloVe Model (we are going to load all 4 versions for comparison)
glove_model_50 = api.load("glove-wiki-gigaword-50")
glove_model_100 = api.load("glove-wiki-gigaword-100")
glove_model_200 = api.load("glove-wiki-gigaword-200")
glove_model_300 = api.load("glove-wiki-gigaword-300")
print("Models loaded successfully!\n")


Models loaded successfully!



In [29]:
def calculate_jaccard_similarity(word1, word2):
    """
    Calculates Jaccard similarity based on character overlap.
    Formula: (Intersection of characters) / (Union of characters)
    """
    set1 = set(word1.lower())
    set2 = set(word2.lower())
    
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    
    return intersection / union if union != 0 else 0



### Understanding the Similarity Metrics

To evaluate how closely related two words are, we are using three distinct metrics. Each metric evaluates a different aspect of word similarity:

#### 1. Cosine Similarity (Semantic Meaning)
* **What it measures:** The cosine of the angle between two word vectors in a multi-dimensional space.
* **How it works:** It focuses strictly on the *direction* the vectors are pointing, regardless of their magnitude (length). 
* **Interpretation:** Scores range from `-1` (perfect opposites) to `1` (identical meaning). A score close to `1` means the words appear in very similar contexts in the training data, indicating high semantic similarity (e.g., "king" and "queen").

#### 2. Jaccard Similarity (Spelling / Lexical Overlap)
* **What it measures:** The similarity between the physical characters that make up the words.
* **How it works:** It is calculated as the **Intersection over Union** of the characters. (Number of shared unique characters divided by the total number of unique characters across both words).
* **Interpretation:** Scores range from `0` (no shared letters) to `1` (exact same letters). This metric is completely blind to meaning; it only cares about spelling (e.g., "bear" and "bare" will score very high).

#### 3. Euclidean Distance (Spatial Distance)
* **What it measures:** The straight-line distance between the endpoints of two word vectors in space.
* **How it works:** Unlike Cosine Similarity, Euclidean Distance factors in both the direction *and* the magnitude (length) of the vectors. 
* **Interpretation:** Because it is a measure of *distance*, a **lower** score means the words are closer together (more similar). A distance of `0.0` means the vectors are exactly on top of each other. Note that as vector dimensions increase (from 50 to 300), Euclidean distances naturally tend to get larger because space expands.

In [30]:
def compare_words(word1, word2, model):
    print(f"--- Comparing: '{word1}' & '{word2}' using {model} ---")
    
    # 2. Calculate Cosine Similarity using GloVe
    try:
        # gensim's .similarity() calculates cosine similarity under the hood
        cosine_sim = model.similarity(word1, word2)
        print(f"Cosine Similarity (Meaning): {cosine_sim:.4f}")
    except KeyError as e:
        print(f"Cosine Similarity Error: One or both words are not in the GloVe vocabulary.")

    # 3. Calculate Jaccard Similarity using characters
    jaccard_sim = calculate_jaccard_similarity(word1, word2)
    print(f"Jaccard Similarity (Spelling): {jaccard_sim:.4f}")

    # 4. Calculate Euclidean Distance between the two word vectors
    try:
        vector1 = model[word1.lower()]
        vector2 = model[word2.lower()]
        
        # Calculate the straight-line distance between the two arrays
        euclidean_dist = np.linalg.norm(vector1 - vector2)
        print(f"Euclidean Distance (Space between vectors): {euclidean_dist:.4f}")
    except KeyError:
        print("Euclidean Distance Error: Words not in vocabulary.")
    print("\n")

In [31]:
# Example A: High meaning overlap, low spelling overlap
compare_words("king", "queen", glove_model_50)
compare_words("king", "queen", glove_model_100)
compare_words("king", "queen", glove_model_200)
compare_words("king", "queen", glove_model_300)

--- Comparing: 'king' & 'queen' using KeyedVectors<vector_size=50, 400000 keys> ---
Cosine Similarity (Meaning): 0.7839
Jaccard Similarity (Spelling): 0.1429
Euclidean Distance (Space between vectors): 3.4778


--- Comparing: 'king' & 'queen' using KeyedVectors<vector_size=100, 400000 keys> ---
Cosine Similarity (Meaning): 0.7508
Jaccard Similarity (Spelling): 0.1429
Euclidean Distance (Space between vectors): 4.2813


--- Comparing: 'king' & 'queen' using KeyedVectors<vector_size=200, 400000 keys> ---
Cosine Similarity (Meaning): 0.6665
Jaccard Similarity (Spelling): 0.1429
Euclidean Distance (Space between vectors): 5.4949


--- Comparing: 'king' & 'queen' using KeyedVectors<vector_size=300, 400000 keys> ---
Cosine Similarity (Meaning): 0.6336
Jaccard Similarity (Spelling): 0.1429
Euclidean Distance (Space between vectors): 5.9663




In [32]:
# Example B: Low meaning overlap, high spelling overlap
compare_words("bear", "bare", glove_model_50)
compare_words("bear", "bare", glove_model_100)
compare_words("bear", "bare", glove_model_200)
compare_words("bear", "bare", glove_model_300)

--- Comparing: 'bear' & 'bare' using KeyedVectors<vector_size=50, 400000 keys> ---
Cosine Similarity (Meaning): 0.4236
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 4.7043


--- Comparing: 'bear' & 'bare' using KeyedVectors<vector_size=100, 400000 keys> ---
Cosine Similarity (Meaning): 0.3773
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 5.7530


--- Comparing: 'bear' & 'bare' using KeyedVectors<vector_size=200, 400000 keys> ---
Cosine Similarity (Meaning): 0.2219
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 7.3358


--- Comparing: 'bear' & 'bare' using KeyedVectors<vector_size=300, 400000 keys> ---
Cosine Similarity (Meaning): 0.1231
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 8.2707




In [33]:
# Example C: Identical words
compare_words("apple", "apple", glove_model_50)
compare_words("apple", "apple", glove_model_100)
compare_words("apple", "apple", glove_model_200)
compare_words("apple", "apple", glove_model_300)

--- Comparing: 'apple' & 'apple' using KeyedVectors<vector_size=50, 400000 keys> ---
Cosine Similarity (Meaning): 1.0000
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 0.0000


--- Comparing: 'apple' & 'apple' using KeyedVectors<vector_size=100, 400000 keys> ---
Cosine Similarity (Meaning): 1.0000
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 0.0000


--- Comparing: 'apple' & 'apple' using KeyedVectors<vector_size=200, 400000 keys> ---
Cosine Similarity (Meaning): 1.0000
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 0.0000


--- Comparing: 'apple' & 'apple' using KeyedVectors<vector_size=300, 400000 keys> ---
Cosine Similarity (Meaning): 1.0000
Jaccard Similarity (Spelling): 1.0000
Euclidean Distance (Space between vectors): 0.0000




### Inference

- The semantic similarity comparison shows that `king` and `queen` are strongly related in meaning across GloVe sizes, with high cosine similarity and relatively small Euclidean distance, despite low Jaccard overlap from different spelling.

- The pair `bear` and `bare` demonstrates the opposite: high Jaccard similarity from shared characters, but low cosine similarity and larger Euclidean distance, indicating low semantic similarity.

- The identical pair `apple` and `apple` produces the expected result: maximum semantic similarity, maximum Jaccard similarity, and zero Euclidean distance.

- Overall, the different GloVe dimensions (50, 100, 200, 300) show consistent behavior for these examples, confirming that character-based similarity and embedding-based semantic similarity capture different aspects of word relatedness.